In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple

In [ ]:
OUTPUT_DIR = Path.cwd()
RESULTS_A_PATH = OUTPUT_DIR / 'results_A.csv'
RESULTS_B_PATH = OUTPUT_DIR / 'results_B.csv'
RESULTS_C_PATH = OUTPUT_DIR / 'results_C.csv'
BENCHMARKS_A_PATH = OUTPUT_DIR / 'benchmarks_A.csv'
BENCHMARKS_B_PATH = OUTPUT_DIR / 'benchmarks_B.csv'
FIG1_PLOTS_PATH = OUTPUT_DIR / 'fig1_plots.pdf'
FIG2_PLOTS_PATH = OUTPUT_DIR / 'fig2_plot.pdf'

In [ ]:
def plot_A(results_df, benchmarks_df, ax, estimator, color): # minimal variation due to sampling error, omitting population-limit by default
    x = results_df['m'].unique()*2
    label = estimator.lower() + '_regret'
    y_emp = results_df[results_df['n'] != -1].groupby('m')[label]
    y_emp_mean = y_emp.mean()
    y_emp_min = y_emp.min()
    y_emp_max = y_emp.max()
    ax.plot(x, y_emp_mean, color=color)
    ax.fill_between(
        x,
        y_emp_min,
        y_emp_max,
        color=color,
        alpha=0.2,
        linewidth=0,
    )

    x = benchmarks_df["m"].unique()*2
    y = benchmarks_df[benchmarks_df["method"] == estimator]["regret"]
    ax.plot(x, y, '--', color=color)

def plot_B(results_df, benchmarks_df, ax, estimator, color, show_pop=False):
    x = results_df["eps_hat"].unique()
    label = estimator.lower() + '_regret'
    y_emp = results_df[results_df["n"] != -1].groupby("eps_hat")[label]
    y_emp_mean = y_emp.mean()
    y_emp_min = y_emp.min()
    y_emp_max = y_emp.max()
    ax.plot(x, y_emp_mean, color=color)
    ax.fill_between(
        x,
        y_emp_min,
        y_emp_max,
        color=color,
        alpha=0.2,
        linewidth=0,
    )

    x = benchmarks_df["rho"].unique()
    y = benchmarks_df[benchmarks_df["method"] == estimator]["regret"]
    ax.plot(x, y,  '--',color=color)

def plot_C(results_df, ax, estimator, color):
    x = results_df['effective_rank'].unique()
    label = estimator.lower() + '_regret'
    y_emp = results_df[results_df['n'] != -1].groupby('effective_rank')[label]
    y_emp_mean = y_emp.mean()
    y_emp_min = y_emp.min()
    y_emp_max = y_emp.max()
    ax.plot(x, y_emp_mean, color=color)
    ax.fill_between(
        x,
        y_emp_min,
        y_emp_max,
        color=color,
        alpha=0.2,
        linewidth=0,
    )

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase

class HandlerHalfSolidHalfDashed(HandlerBase):
    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height,
                       fontsize, trans):
        color = orig_handle.get_color()
        lw = orig_handle.get_linewidth()

        y = 0.5 * height - ydescent
        x0 = -xdescent
        xm = x0 + 0.5 * width
        x1 = x0 + width

        left = Line2D(
            [x0, xm], [y, y],
            color=color,
            lw=lw,
            linestyle="-",
            solid_capstyle="butt",
            transform=trans,
        )

        dash_len = 1.4
        gap_len = 1.4
        leading_gap = 1
        dash_offset = dash_len + gap_len - leading_gap

        right = Line2D(
            [xm, x1], [y, y],
            color=color,
            lw=lw,
            linestyle=(dash_offset, (dash_len, gap_len)),
            dash_capstyle="butt",
            transform=trans,
        )

        return [left, right]

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})

fig, axs = plt.subplots(1, 2, figsize=(7.5,3), constrained_layout=True, gridspec_kw={"wspace": 0.07})

colors = [
    "#D55E00",  # vermillion
    "#E69F00",  # orange
    "#0072B2",  # blue
    "#009E73",  # bluish green
]
estimators = ['ERM', 'DRO', 'MRO', 'SNR']

ax = axs[0]
results_df = pd.read_csv(RESULTS_A_PATH)
benchmarks_df = pd.read_csv(BENCHMARKS_A_PATH)
for i,estimator in enumerate(estimators):
    plot_A(results_df, benchmarks_df, ax, estimator, colors[i])
ax.set_xlabel(r"dimension $d$")
ax.set_ylabel("population regret")
ax.set_title("Setting A: Regret vs. Dimension")

handles = []
labels = []
for color, estimator in zip(colors, estimators):
    dummy = Line2D([], [], color=color, lw=2)
    handles.append(dummy)
    labels.append(estimator)

ax.legend(handles, labels,
          handler_map={Line2D: HandlerHalfSolidHalfDashed()},
          handlelength=3,
          handletextpad=0.8,)

ax = axs[1]
results_df = pd.read_csv(RESULTS_B_PATH)
benchmarks_df = pd.read_csv(BENCHMARKS_B_PATH)
for i,estimator in enumerate(estimators):
    plot_B(results_df, benchmarks_df, ax, estimator, colors[i])
ax.set_xlabel(r"supplied radius $\rho$")
ax.set_xscale("log")
ax.set_ylabel("population regret")
ax.set_title("Setting B: Regret vs. Supplied Radius")

ax.axvline(
    x=1,
    linestyle="--",
    color="grey",
    linewidth=1,
)

fig.savefig(FIG1_PLOTS_PATH, bbox_inches="tight")

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})

fig, ax = plt.subplots(1, 1, figsize=(3.5,3), constrained_layout=True, gridspec_kw={"wspace": 0.07})

colors = [
    "#D55E00",  # vermillion
    "#E69F00",  # orange
    "#0072B2",  # blue
    "#009E73",  # bluish green
]
estimators = ['ERM', 'DRO', 'MRO', 'SNR']

results_df = pd.read_csv(RESULTS_C_PATH)
for i,estimator in enumerate(estimators):
    plot_C(results_df, ax, estimator, colors[i])
ax.set_xlabel(r"effective rank $r$")
ax.set_ylabel("population regret")
ax.set_title("Setting C: Regret vs. Effective Rank")

handles = [
    Line2D([], [], color=color, lw=2, linestyle="-")
    for color in colors
]

ax.legend(
    handles,
    estimators,
    handlelength=2,
    handletextpad=0.8,
)

fig.savefig(FIG2_PLOTS_PATH, bbox_inches="tight")